# C-SPIKES: fluorescence to spikes
Select **Runtime → Change runtime type → GPU**, then **Run all**. Use a Python 3.13 Linux runtime. No Drive mount or companion Python files are needed.

This example installs published **c-spikes 0.2.0rc1** wheels into an isolated environment, fetches four repository assets, and runs PGAS plus a pretrained BiophysML/ENS2 model on the same ten-second recording. The first installation downloads several GB; allow several minutes and approximately 10 GB of free disk. GPU availability and speed vary. The inference cells stop on errors rather than silently falling back to CPU.

The window is deliberately unchanged from the initial demo: cell01, epoch0, first10 seconds. It contains **no recorded action potentials** under the supplied timebase. This illustrates execution and interpretation, not model accuracy, ranking or convergence. PGAS uses 1,000 particles and 200 iterations (100 burn-in); ENS2 uses the selected December checkpoint. Live training and calibration are not included.

Data: [Janelia/DANDI 000168](https://dandiarchive.org/dandiset/000168/draft), Rozsa et al.; jGCaMP8, Zhang et al. Upstream data are CC BY4.0; the exact processed-MAT mapping still needs documentation. Checkpoint training overlap and redistribution terms remain to be documented before broad distribution. This review notebook uses the repository owner's selected checkpoint; it does not establish that it is the best-performing model.


## 1. Choose the source and create a fresh workspace
During review, the notebook and asset fetch both target `review/colab-standalone`. That branch must be pushed before the public Colab link works. After review/merge, update the link and `REPO_REF` together. Assets are checked against the hashes below, so unexpected changes stop execution. The checkout supplies assets only; Python imports come from PyPI.


In [ ]:
import os, sys, json, hashlib, subprocess, tempfile, time, platform, shutil
from pathlib import Path

REPO_URL = "https://github.com/gerardjb/C-SPIKES.git"
REPO_REF = "review/colab-standalone"
PIP_SOURCE = ["--index-url", "https://pypi.org/simple"]
PACKAGES = ["c-spikes[ens2]==0.2.0rc1", "numpy==2.1.3", "scipy==1.16.3",
            "torch==2.11.0", "matplotlib==3.10.0", "h5py==3.14.0"]
assert sys.version_info[:2] == (3, 13), "Select a Python 3.13 runtime for this demo."
assert platform.system() == "Linux" and platform.machine() == "x86_64", "Use the Linux x86_64 GPU runtime."
ROOT = Path(tempfile.mkdtemp(prefix="cspikes-demo-", dir=Path.cwd())).resolve()
REPO, OUT = ROOT / "assets", ROOT / "results"
OUT.mkdir()
ASSETS = {
 "data": ("sample_data/janelia_8f/excitatory/jGCaMP8f_ANM471993_cell01.mat", "4395807cca1d6bc342d1f720169256733ced25b6090a61c4ec1ddbe745d24569"),
 "constants": ("parameter_files/constants_GCaMP8_soma.json", "cbab4a5aa06004ec9e814b465a7870cdcadbbd80d621c89eb8d0039b28e965e7"),
 "sensor": ("src/c_spikes/pgas/20251210_Janelia_8f_params.dat", "7b6a089fcc68611798e4473731f7ecc25f206cceaeae24ebdbb30c9e4acc6e03"),
 "model": ("Pretrained_models/BiophysML/ens2_synth_comparison_k1_r12_s2p0_d0p45_sb11000/exc_ens2_pub.pt", "3ed1a57fa13a3b5782ded277a5ac295351f8d6a265f4d424707235ad18b2e9aa"),
}
print("Workspace:", ROOT)


In [ ]:
subprocess.run(["git", "clone", "--filter=blob:none", "--depth=1", "--single-branch",
                "--no-checkout", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True, timeout=180)
patterns = "\n".join("/" + path for path, _ in ASSETS.values()) + "\n/LICENSE\n"
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", "--no-cone", "--stdin"],
               input=patterns, text=True, check=True, timeout=60)
subprocess.run(["git", "-C", str(REPO), "checkout"], check=True, timeout=180)
source_sha = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
for name, (path, expected) in ASSETS.items():
    actual = hashlib.sha256((REPO / path).read_bytes()).hexdigest()
    assert actual == expected, f"Asset changed: {name}; review before proceeding."
provenance = {"repository": REPO_URL, "requested_ref": REPO_REF, "resolved_commit": source_sha,
              "assets": ASSETS, "packages_requested": PACKAGES, "python": sys.version,
              "platform": platform.platform()}
(OUT / "source.json").write_text(json.dumps(provenance, indent=2))
print("Verified source and four assets:", source_sha)


## 2. Install published wheels
`pip` selects the wheel for Python, ABI and operating system—not for the GPU model. The Linux wheel contains CPU and CUDA backends. GPU usability is checked separately below. Installation is binary-only and never builds C-SPIKES from the checkout.

The isolated environment avoids changing Colab's preinstalled packages. Each inference call runs in a fresh process so native PGAS and PyTorch do not retain competing CUDA contexts. All Python code is visible in this notebook; no downloaded helper scripts are executed.


In [ ]:
VENV = ROOT / "venv"
subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True, timeout=60)
PYTHON = str(VENV / "bin/python")
ENV = os.environ.copy()
for key in ("PYTHONPATH", "PYTHONHOME", "LD_PRELOAD", "LD_LIBRARY_PATH",
            "C_SPIKES_PGAS_RESAMPLING", "C_SPIKES_PGAS_TRAJECTORY", "C_SPIKES_PGAS_ANCESTOR_SEED",
            "PIP_INDEX_URL", "PIP_EXTRA_INDEX_URL", "PIP_NO_INDEX", "PIP_FIND_LINKS"):
    ENV.pop(key, None)
ENV.update(C_SPIKES_PGAS_BACKEND="cuda", PYTHONNOUSERSITE="1", OMP_NUM_THREADS="1",
           OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1", MPLBACKEND="Agg",
           PIP_CONFIG_FILE=os.devnull, PIP_DISABLE_PIP_VERSION_CHECK="1",
           CSP_DEMO_ROOT=str(ROOT))
subprocess.run([PYTHON, "-m", "pip", "install", *PIP_SOURCE, "--only-binary=:all:",
                "--report", str(OUT / "pip-install-report.json"), *PACKAGES],
               env=ENV, check=True, timeout=900)
subprocess.run([PYTHON, "-m", "pip", "check"], env=ENV, check=True, timeout=30)
freeze = subprocess.check_output([PYTHON, "-m", "pip", "freeze", "--all"], env=ENV, text=True)
(OUT / "environment.txt").write_text(freeze)
report = json.loads((OUT / "pip-install-report.json").read_text())
wheel = next(x for x in report["install"] if x["metadata"]["name"].lower().replace("_", "-") == "c-spikes")
assert wheel["download_info"]["archive_info"]["hashes"]["sha256"] == "75cd69888c695d71f780b936ac0827a7631622e05a31c719a27f0ad78d51328a", "Unexpected C-SPIKES wheel."
print("Published wheel verified; environment ready.")


In [ ]:
GPU_CHECK = r"""
import os,json,sys
from pathlib import Path
from importlib import metadata
import torch
root=Path(os.environ["CSP_DEMO_ROOT"])
assert torch.cuda.is_available(), "PyTorch cannot use the assigned GPU."
assert torch.cuda.device_count()==1, "This demo expects one visible GPU."
assert torch.cuda.get_device_capability(0)>=(7,5), "The wheel requires Turing (T4) or newer."
x=torch.ones(8,device="cuda"); assert float(x.sum().cpu())==8
info={"torch":torch.__version__,"torch_cuda":torch.version.cuda,"cudnn":torch.backends.cudnn.version(),
      "gpu":torch.cuda.get_device_name(0),"compute_capability":torch.cuda.get_device_capability(0),
      "packages":{n:metadata.version(n) for n in ("c-spikes","numpy","scipy","torch","matplotlib","h5py")}}
(root/"results/runtime.json").write_text(json.dumps(info,indent=2));print(json.dumps(info,indent=2))
"""
subprocess.run([PYTHON, "-c", GPU_CHECK], env=ENV, cwd=OUT, check=True, timeout=60)
subprocess.run([PYTHON, "-c", "from c_spikes.pgas import pgas_bound_gpu; print('Native CUDA backend loaded:', pgas_bound_gpu.__file__)"],
               env=ENV, cwd=OUT, check=True, timeout=60)


## 3. Prepare the shared input
Load the first ten seconds without baseline correction, filtering or resampling. Ground-truth AP times are used only for plotting. The parameter file stays unchanged except for sampling frequency, iteration count and seed. ENS2's existing adapter performs its own model-required resampling.


In [ ]:
PREPARE = r"""
import os,json,hashlib
from pathlib import Path
import numpy as np
from scipy.io import loadmat
root=Path(os.environ["CSP_DEMO_ROOT"]);out=root/"results";repo=root/"assets"
a=json.loads((out/"source.json").read_text())["assets"]
d=loadmat(repo/a["data"][0]);times=np.asarray(d["time_stamps"],dtype=float);values=np.asarray(d["dff"],dtype=float)
assert times.ndim==2 and times.shape==values.shape
end=np.flatnonzero(times[0]>=times[0,0]+10);assert end.size
n=int(end[0]);t=times[0,:n].copy();y=values[0,:n].copy()
assert n>=96 and np.isfinite(t).all() and np.isfinite(y).all()
dt=np.diff(t);assert (dt>0).all() and np.allclose(dt,np.median(dt),rtol=.01,atol=1e-8)
gt=np.asarray(d["ap_times"],dtype=float).ravel();assert np.isfinite(gt).all()
gt=gt[(gt>=t[0])&(gt<t[0]+10)]
np.savez(out/"input.npz",time=t,fluorescence=y)
np.save(out/"recorded_APs.npy",gt,allow_pickle=False)
config=json.loads((repo/a["constants"][0]).read_text())
assert config["MCMC"]["nparticles"]==1000 and config["ODE"]["time step mode"]==1
config["data"]["sampling_frequency"]=float(1/np.median(dt));config["MCMC"].update(niter=200,seed=11)
(out/"effective_constants.json").write_text(json.dumps(config,indent=2))
info={"samples":n,"start":float(t[0]),"end":float(t[-1]),"sampling_frequency":float(1/np.median(dt)),
      "recorded_AP_count":int(gt.size),"input_sha256":hashlib.sha256((out/"input.npz").read_bytes()).hexdigest(),
      "iterations":200,"burnin":100,"seed":11,"particles":1000}
(out/"input.json").write_text(json.dumps(info,indent=2));print(info)
"""
subprocess.run([PYTHON, "-c", PREPARE], env=ENV, cwd=OUT, check=True, timeout=30)


## 4. Run PGAS on the GPU
This is the slowest step. It has a 20-minute watchdog, retains all 200 iterations and excludes the first100 from the displayed posterior summary. It does not shorten the workload to make a slow GPU appear to pass. Logs and partial outputs remain available if it stops.


In [ ]:
PGAS = r"""
import os,json,time,hashlib
from pathlib import Path
import numpy as np
from c_spikes.pgas import pgas_bound_gpu as native
from c_spikes.inference.pgas import load_pgas_method_result
from c_spikes.inference.types import TrialSeries
root=Path(os.environ["CSP_DEMO_ROOT"]);out=root/"results";dest=out/"pgas";dest.mkdir()
a=json.loads((out/"source.json").read_text())["assets"]
with np.load(out/"input.npz",allow_pickle=False) as f:t=f["time"].copy();y=f["fluorescence"].copy()
start=time.monotonic()
analyzer=native.Analyzer(time=t,data=y,constants_file=str(out/"effective_constants.json"),
 output_folder=str(dest),column=1,tag="demo_trial0",niter=200,trainedPriorFile="",append=False,trim=1,
 verbose=False,gtSpikes=np.empty(0),has_trained_priors=False,has_gtspikes=False,maxlen=len(t),
 Gparam_file=str(root/"assets"/a["sensor"][0]),seed=11)
analyzer.run()
assert analyzer.get_run_status()=="completed",analyzer.get_run_message()
logp=np.atleast_1d(np.loadtxt(dest/"logp_demo_trial0.dat"));assert logp.size==200 and np.isfinite(logp).all()
params=np.asarray(analyzer.get_parameter_samples());assert params.shape==(200,6) and np.isfinite(params).all()
np.save(dest/"parameter_samples.npy",params,allow_pickle=False)
r=load_pgas_method_result([TrialSeries(times=t,values=y)],"demo",dest,100)
tt=np.asarray(r.time_stamps).ravel();pp=np.asarray(r.spike_prob).ravel()
assert np.array_equal(tt,t) and pp.shape==t.shape and np.isfinite(pp).all()
payload={"time":tt,"prediction":pp}
for name in ("reconstruction","discrete_spikes"):
 v=getattr(r,name,None)
 if v is not None:
  v=np.asarray(v).ravel();assert v.shape==pp.shape and np.isfinite(v).all();payload[name]=v
np.savez(dest/"result.npz",**payload)
(dest/"metadata.json").write_text(json.dumps({"seconds":time.monotonic()-start,"iterations":200,"burnin":100,
 "input_sha256":hashlib.sha256((out/"input.npz").read_bytes()).hexdigest(),"adapter":r.metadata},indent=2))
print("PGAS completed: 200 iterations, finite posterior and full-length output.",flush=True)
"""
started = time.monotonic()
with (OUT / "pgas.log").open("x") as log:
    process = subprocess.Popen([PYTHON, "-u", "-c", PGAS], env=ENV, cwd=OUT, stdout=log, stderr=subprocess.STDOUT)
    try:
        rc = process.wait(timeout=1200)
    finally:
        if process.poll() is None:
            process.kill()
            process.wait()
print((OUT / "pgas.log").read_text()[-4000:])
assert rc == 0, "Inference failed; inspect the saved log."
print("Elapsed seconds:", round(time.monotonic() - started, 2))


## 5. Run the pretrained BiophysML/ENS2 model
Use the owner's selected December excitatory checkpoint. Its exact SHA256 is checked again immediately before loading: PyTorch checkpoints must come from a trusted source. This is inference only, not training. The existing adapter determines preprocessing, output timebase and discretization.


In [ ]:
ENS2 = r"""
import os,json,time,hashlib
from pathlib import Path
import numpy as np
import torch
from c_spikes.inference.ens2 import Ens2Config,run_ens2_inference
from c_spikes.inference.cache import set_cache_root
root=Path(os.environ["CSP_DEMO_ROOT"]);out=root/"results";dest=out/"ens2";dest.mkdir()
a=json.loads((out/"source.json").read_text())["assets"];model=root/"assets"/a["model"][0]
assert hashlib.sha256(model.read_bytes()).hexdigest()==a["model"][1]
assert torch.cuda.is_available(),"No silent CPU fallback."
set_cache_root(root/"inference-cache")
with np.load(out/"input.npz",allow_pickle=False) as f:t=f["time"].copy();y=f["fluorescence"].copy()
start=time.monotonic()
r=run_ens2_inference(t[None,:],y[None,:],Ens2Config(dataset_tag="colab-demo",pretrained_dir=model.parent,neuron_type="Exc",use_cache=False))
tt=np.asarray(r.time_stamps).ravel();pp=np.asarray(r.spike_prob).ravel()
assert len(tt)>0 and tt.shape==pp.shape and np.isfinite(tt).all() and np.isfinite(pp).all()
assert (np.diff(tt)>0).all() and tt[0]==t[0] and tt[-1]==t[-1]
payload={"time":tt,"prediction":pp}
for name in ("reconstruction","discrete_spikes"):
 v=getattr(r,name,None)
 if v is not None:
  v=np.asarray(v).ravel();assert v.shape==pp.shape and np.isfinite(v).all();payload[name]=v
np.savez(dest/"result.npz",**payload)
(dest/"metadata.json").write_text(json.dumps({"seconds":time.monotonic()-start,"checkpoint_sha256":a["model"][1],
 "input_sha256":hashlib.sha256((out/"input.npz").read_bytes()).hexdigest(),"adapter":r.metadata},indent=2))
print("Pretrained inference completed:",len(pp),"finite predictions.",flush=True)
"""
started = time.monotonic()
with (OUT / "ens2.log").open("x") as log:
    process = subprocess.Popen([PYTHON, "-u", "-c", ENS2], env=ENV, cwd=OUT, stdout=log, stderr=subprocess.STDOUT)
    try:
        rc = process.wait(timeout=300)
    finally:
        if process.poll() is None:
            process.kill()
            process.wait()
print((OUT / "ens2.log").read_text()[-4000:])
assert rc == 0, "Inference failed; inspect the saved log."
print("Elapsed seconds:", round(time.monotonic() - started, 2))


## 6. Inspect and save the results
PGAS reports mean spike counts per acquisition bin. ENS2 reports its adapter's rate output on a different sampling grid. They are shown on separate axes, not treated as equivalent probabilities or compared by raw height. A log-probability trace is included; a completed run is not proof of convergence.

Download the ZIP before disconnecting: Colab runtime files are temporary. It includes the input, effective configuration, source hashes, installation report, environment, predictions, native posterior files and plots, but no model weights. No data are uploaded to an external service.


In [ ]:
PLOT = r"""
import os,json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
root=Path(os.environ["CSP_DEMO_ROOT"]);out=root/"results"
with np.load(out/"input.npz",allow_pickle=False) as f:t=f["time"].copy();y=f["fluorescence"].copy()
gt=np.load(out/"recorded_APs.npy",allow_pickle=False)
fig,axes=plt.subplots(4,1,sharex=True,figsize=(12,8));axes[0].plot(t,y);axes[0].set_ylabel("Observed ΔF/F")
axes[1].vlines(gt,0,1);axes[1].set_ylabel("Recorded APs")
for ax,name,label in zip(axes[2:],["pgas","ens2"],["PGAS mean count/bin","ENS2 rate output"]):
 with np.load(out/name/"result.npz",allow_pickle=False) as f:ax.plot(f["time"],f["prediction"])
 ax.set_ylabel(label)
axes[-1].set_xlabel("Time (s)");fig.suptitle("Live inference on a quiet example window — not an accuracy benchmark")
fig.tight_layout();fig.savefig(out/"predictions.png",dpi=140);plt.close(fig)
logp=np.atleast_1d(np.loadtxt(out/"pgas/logp_demo_trial0.dat"));fig,ax=plt.subplots(figsize=(12,3))
ax.plot(np.arange(len(logp)),logp);ax.axvline(99.5,color="black",linestyle="--")
ax.set(xlabel="Saved iteration (zero based)",ylabel="Native log probability")
fig.tight_layout();fig.savefig(out/"posterior_trace.png",dpi=140);plt.close(fig)
a=json.loads((out/"pgas/metadata.json").read_text());b=json.loads((out/"ens2/metadata.json").read_text())
assert a["input_sha256"]==b["input_sha256"]
(out/"completed.json").write_text(json.dumps({"status":"passed","same_input_sha256":a["input_sha256"],"pgas_iterations":200,"pgas_seconds":a["seconds"],"ens2_seconds":b["seconds"]},indent=2))
"""
subprocess.run([PYTHON, "-c", PLOT], env=ENV, cwd=OUT, check=True, timeout=60)
from IPython.display import Image, display
for filename in ("predictions.png", "posterior_trace.png"):
    display(Image(filename=str(OUT / filename)))
print((OUT / "completed.json").read_text())


In [ ]:
import zipfile
archive = shutil.make_archive(str(ROOT / "cspikes-demo-results"), "zip", OUT)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
print("Results:", archive)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: download the ZIP from the printed path.")
else:
    files.download(archive)
